In [1]:
import os, re
from typing import List, Dict, Optional
import time
from sklearn.model_selection import StratifiedKFold, GridSearchCV

In [2]:
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

In [3]:
from transformers import AutoTokenizer, AutoModel, AutoConfig, T5EncoderModel
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, average_precision_score, make_scorer
)

In [4]:
def safe_model_tag(model_id: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", model_id)

In [5]:
def clean_aa(seq: str) -> str:
    # keep only standard AA-ish; map rare to X
    return seq.strip().upper().replace("U","X").replace("Z","X").replace("O","X").replace("B","X")

In [6]:
def needs_space_separated(model_id: str) -> bool:
    mid = model_id.lower()
    return any(k in mid for k in ["rostlab", "prot_t5", "prot_bert", "distilprotbert"])

In [ ]:
def mean_pool(last_hidden: torch.Tensor,
              attn_mask: torch.Tensor,
              special_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    mask = attn_mask.bool()

    if special_mask is not None:
        keep = mask & (~special_mask.bool())
        bad = (keep.sum(dim=1) == 0)
        if bad.any():
            keep[bad] = mask[bad]
        mask = keep

    mask_f = mask.unsqueeze(-1).float()            # [B, L, 1]
    summed = (last_hidden * mask_f).sum(dim=1)     # [B, H]
    denom = mask_f.sum(dim=1).clamp(min=1.0)       # [B, 1]
    return summed / denom

In [8]:
def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")

    ap = average_precision_score(y_true, y_prob)
    return {"ACC": acc, "BACC": bacc, "Sn": sn, "Sp": sp, "MCC": mcc, "AUC": auc, "AP": ap}

In [ ]:
def load_model_tokenizer_transformers(model_id: str, device: str):
    mid = model_id.lower()

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    if "mistral-prot" in mid:
        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype="auto",
        ).to(device).eval()
        return tok, model

    if mid.startswith("synthyra/ankh"):
        model = T5EncoderModel.from_pretrained(
            model_id,
            torch_dtype=torch.float32,
        ).to(device).eval()
        return tok, model

    cfg = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    torch_dtype = torch.float16 if str(device).startswith("cuda") else torch.float32

    if getattr(cfg, "model_type", "") == "t5":
        model = T5EncoderModel.from_pretrained(model_id, torch_dtype=torch_dtype)
    else:
        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype=torch_dtype,
        )

    model.to(device).eval()
    return tok, model

In [ ]:
def embed_sequences_esmc(model_id: str, seqs: List[str], device: str) -> np.ndarray:
    if "300m" in model_id.lower():
        esm_name = "esmc_300m"
    elif "600m" in model_id.lower():
        esm_name = "esmc_600m"
    else:
        raise ValueError(f"Unknown ESMC size in model_id: {model_id}")

    from esm.models.esmc import ESMC
    from esm.sdk.api import ESMProtein, LogitsConfig

    client = ESMC.from_pretrained(esm_name).to(device)
    client.eval()

    vecs = []
    with torch.no_grad():
        for seq in tqdm(seqs, desc=f"Embedding (ESMC:{esm_name})"):
            s = clean_aa(seq)
            protein = ESMProtein(sequence=s)
            protein_tensor = client.encode(protein)
            out = client.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))

            emb = out.embeddings
            seq_emb = getattr(emb, "sequence", None)
            if seq_emb is None and isinstance(emb, dict):
                seq_emb = emb.get("sequence", None)
            if seq_emb is None:
                seq_emb = emb

            if not torch.is_tensor(seq_emb):
                seq_emb = torch.tensor(seq_emb)

            if seq_emb.dim() == 3:
                seq_emb = seq_emb[0]  # [L, D]

            if seq_emb.size(0) == len(s) + 2:
                seq_emb = seq_emb[1:-1]

            vec = seq_emb.mean(dim=0)
            vecs.append(vec.float().cpu().numpy())

    X = np.stack(vecs, axis=0)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
def embed_sequences(model_id: str,
                    seqs: List[str],
                    device: str,
                    batch_size: int,
                    cache_dir: str,
                    max_length: int) -> np.ndarray:
    os.makedirs(cache_dir, exist_ok=True)
    tag = safe_model_tag(model_id)
    cache_path = os.path.join(cache_dir, f"{tag}.npy")

    if os.path.exists(cache_path):
        return np.load(cache_path)

    if model_id.startswith("EvolutionaryScale/esmc-"):
        X = embed_sequences_esmc(model_id, seqs, device)

    np.save(cache_path, X)
    return X

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

def solver_cv_select_strict_auc(
    X_tr,
    y_tr,
    seed=52,
    out_csv="sklearn_cv_EvolutionaryScale_esmc-600m-2024-12_seed42_test0.2.csv",
    n_jobs=8,
    models=("logreg", "rf", "et"),
):
    scoring = {
        "mcc": make_scorer(matthews_corrcoef),
        "auc": "roc_auc",
        "bacc": "balanced_accuracy",
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    # 注意：
    # 1. logreg 需要 scaler
    # 2. tree 类模型一般不需要 scaler
    # 3. RF/ET 里 n_jobs 建议设为 1，避免和 GridSearchCV 的 n_jobs 嵌套并行打架
    model_spaces = {
        "logreg": {
            "pipeline": Pipeline(steps=[
                ("imp", SimpleImputer(strategy="constant", fill_value=0.0)),
                ("scaler", MinMaxScaler()),
                ("clf", LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=seed
                ))
            ]),
            "param_grid": [
                {"clf__solver": ["saga"], "clf__penalty": ["l2"], "clf__C": [1]},
            ],
        },

        "rf": {
            "pipeline": Pipeline(steps=[
                ("imp", SimpleImputer(strategy="constant", fill_value=0.0)),
                ("clf", RandomForestClassifier(
                    class_weight="balanced",
                    random_state=seed,
                    n_jobs=1
                ))
            ]),
            "param_grid": [
                {
                    "clf__n_estimators": [200],
                    "clf__max_depth": [None],
                    "clf__min_samples_split": [2],
                    "clf__min_samples_leaf": [1],
                    "clf__max_features": ["sqrt"],
                }
            ],
        },

        "et": {
            "pipeline": Pipeline(steps=[
                ("imp", SimpleImputer(strategy="constant", fill_value=0.0)),
                ("clf", ExtraTreesClassifier(
                    class_weight="balanced",
                    random_state=seed,
                    n_jobs=1
                ))
            ]),
            "param_grid": [
                {
                    "clf__n_estimators": [200],
                    "clf__max_depth": [None],
                    "clf__min_samples_split": [2],
                    "clf__min_samples_leaf": [1],
                    "clf__max_features": ["sqrt"],
                }
            ],
        },
    }

    rows = []
    best_overall = None
    best_info = {
        "best_model_family": None,
        "best_params": None,
        "cv_best_auc": -1e18
    }

    for model_name in models:
        if model_name not in model_spaces:
            raise ValueError(f"Unsupported model: {model_name}. Available: {list(model_spaces.keys())}")

        cfg = model_spaces[model_name]
        t0 = time.time()

        gs = GridSearchCV(
            estimator=cfg["pipeline"],
            param_grid=cfg["param_grid"],
            scoring=scoring,
            refit="auc",
            cv=cv,
            n_jobs=n_jobs,
            verbose=0,
            return_train_score=False
        )
        gs.fit(X_tr, y_tr)

        best_i = gs.best_index_
        n_splits = cv.get_n_splits(X_tr, y_tr)

        fold_auc  = np.array([gs.cv_results_[f"split{k}_test_auc"][best_i]  for k in range(n_splits)], dtype=float)
        fold_mcc  = np.array([gs.cv_results_[f"split{k}_test_mcc"][best_i]  for k in range(n_splits)], dtype=float)
        fold_bacc = np.array([gs.cv_results_[f"split{k}_test_bacc"][best_i] for k in range(n_splits)], dtype=float)

        best_auc  = float(gs.best_score_)
        best_mcc  = float(gs.cv_results_["mean_test_mcc"][best_i])
        best_bacc = float(gs.cv_results_["mean_test_bacc"][best_i])

        dt = time.time() - t0

        row = {
            "model_family": model_name,
            "cv_best_auc": best_auc,
            "cv_mcc_at_best_auc": best_mcc,
            "cv_bacc_at_best_auc": best_bacc,
            "best_params": str(gs.best_params_),
            **{f"best_{k}": v for k, v in gs.best_params_.items()},
            "seconds": dt,

            "fold_auc": ",".join(f"{v:.6f}" for v in fold_auc),
            "fold_mcc": ",".join(f"{v:.6f}" for v in fold_mcc),
            "fold_bacc": ",".join(f"{v:.6f}" for v in fold_bacc),

            "auc_mean": float(fold_auc.mean()),
            "auc_std": float(fold_auc.std(ddof=1)),
            "mcc_mean": float(fold_mcc.mean()),
            "mcc_std": float(fold_mcc.std(ddof=1)),
            "bacc_mean": float(fold_bacc.mean()),
            "bacc_std": float(fold_bacc.std(ddof=1)),
        }

        rows.append(row)
        print(f"[CV:{model_name}] best_auc={best_auc:.4f} best_params={gs.best_params_} time={dt:.1f}s")

        if best_auc > best_info["cv_best_auc"]:
            best_info["cv_best_auc"] = best_auc
            best_info["best_model_family"] = model_name
            best_info["best_params"] = gs.best_params_
            best_overall = gs.best_estimator_

    df_out = pd.DataFrame(rows).sort_values("cv_best_auc", ascending=False)
    df_out.to_csv(out_csv, index=False)

    print("Saved CV table:", out_csv)
    print("Selected (by AUC):", best_info["best_model_family"], "cv_best_auc=", best_info["cv_best_auc"])

    return best_overall, df_out, best_info

In [ ]:
import argparse

def build_parser():
    ap = argparse.ArgumentParser(prog="plm_lr_benchmark") 
    ap.add_argument("--data_csv", required=True)
    ap.add_argument("--seq_col", default="sequence")
    ap.add_argument("--label_col", default="label")
    ap.add_argument("--model_id", required=True)
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_length", type=int, default=256)
    ap.add_argument("--test_size", type=float, default=0.2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--cache_dir", default="cache")
    ap.add_argument("--n_jobs", type=int, default=8)
    ap.add_argument("--out_dir", default="results")
    return ap

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

def main(argv=None):
    ap = build_parser()
    args = ap.parse_args(argv)
    print(args)

    os.makedirs(args.out_dir, exist_ok=True)

    df = pd.read_csv(args.data_csv)
    if args.seq_col not in df.columns:
        raise KeyError(f"seq_col '{args.seq_col}' not found. Columns={list(df.columns)}")
    if args.label_col not in df.columns:
        raise KeyError(f"label_col '{args.label_col}' not found. Columns={list(df.columns)}")

    seqs = df[args.seq_col].astype(str).tolist()
    y = df[args.label_col].astype(int).to_numpy()

    X = embed_sequences(
        args.model_id,
        seqs,
        args.device,
        args.batch_size,
        args.cache_dir,
        args.max_length
    )
    """
    print("X shape:", X.shape)
    print("X mean std over dims:", X.std(axis=0).mean())
    print("Unique rows (rounded):", np.unique(X.round(6), axis=0).shape[0], "/", X.shape[0])
    """
    # 1) One fixed split (test is held-out and used ONCE)
    idx = np.arange(len(y))
    tr_idx, te_idx = train_test_split(
        idx,
        test_size=args.test_size,
        random_state=args.seed,
        shuffle=True,
        stratify=y
    )

    split_path = os.path.join(args.out_dir, f"split_seed{args.seed}_test{args.test_size}.npz")
    np.savez(split_path, train_idx=tr_idx, test_idx=te_idx)
    print("Saved split:", split_path)

    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    # 2) STRICT model selection (ONLY on training data) using AUC
    tag = safe_model_tag(args.model_id)
    cv_csv = os.path.join(args.out_dir, f"sklearn_cv_{tag}_seed{args.seed}_test{args.test_size}.csv")

    best_pipe, df_cv, best_info = solver_cv_select_strict_auc(
        X_tr, y_tr,
        seed=args.seed,
        out_csv=cv_csv,
        n_jobs=getattr(args, "n_jobs", 8),
        models=("logreg", "rf", "et")
    )

    # 3) Final evaluation on held-out test (used ONCE)
    y_prob = best_pipe.predict_proba(X_te)[:, 1]
    y_pred = best_pipe.predict(X_te)
    """
    print("y positive rate:", y.mean())
    print("pred positive rate:", y_pred.mean(), "prob std:", y_prob.std())
    """
    m = compute_metrics(y_te, y_prob, y_pred)

    out = {
        "model_id": args.model_id,
        "n": int(len(df)),
        "dim": int(X.shape[1]),
        "selection": {
            "metric": "AUC",
            "cv_table_csv": os.path.basename(cv_csv),
            "best_model_family": best_info["best_model_family"],
            "best_params": best_info["best_params"],
            "cv_best_auc": float(best_info["cv_best_auc"]),
        },
        "test_metrics": m,
        "split": {
            "method": "train_test_split",
            "test_size": args.test_size,
            "seed": args.seed
        },
    }

    out_path = os.path.join(args.out_dir, f"sklearn_{tag}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print("Saved:", out_path, flush=True)
    print(json.dumps(out["selection"], ensure_ascii=False, indent=2), flush=True)
    print(json.dumps(m, ensure_ascii=False, indent=2), flush=True)

In [ ]:
import gc

model_configs = [
    {"model_id": "EvolutionaryScale/esmc-600m-2024-12", "batch_size": 2},
]

for i, cfg in enumerate(model_configs, 1):
    model_id = cfg["model_id"]
    batch_size = cfg["batch_size"]

    print(f"\n========== Running {i}/{len(model_configs)}: {model_id} | batch_size={batch_size} ==========")

    main([
        "--data_csv", "data.csv",
        "--seq_col", "sequence",
        "--label_col", "label",
        "--model_id", model_id,
        "--batch_size", str(batch_size),
        "--max_length", "256",
    ])
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


========== Running 1/1: EvolutionaryScale/esmc-600m-2024-12 | batch_size=2 ==========
Namespace(data_csv='data.csv', seq_col='sequence', label_col='label', model_id='EvolutionaryScale/esmc-600m-2024-12', device='cuda', batch_size=2, max_length=256, test_size=0.2, seed=42, cache_dir='cache', n_jobs=8, out_dir='results')
X shape: (3444, 1152)
X mean std over dims: 0.014528315
Unique rows (rounded): 3444 / 3444
Saved split: results\split_seed42_test0.2.npz
[CV:logreg] best_auc=0.8193 best_params={'clf__C': 1, 'clf__penalty': 'l2', 'clf__solver': 'saga'} time=53.6s
[CV:rf] best_auc=0.8204 best_params={'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2, 'clf__n_estimators': 200} time=44.4s
[CV:et] best_auc=0.8317 best_params={'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2, 'clf__n_estimators': 200} time=4.9s
Saved CV table: results\sklearn_cv_EvolutionaryScale_esmc-600m-2